<a href="https://colab.research.google.com/github/cjd23/290_audit/blob/main/citysurfaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Thu Oct 29 18:47:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%cd /content
!git clone https://github.com/VIDA-NYU/city-surfaces.git
%cd /content/city-surfaces
!pip install -q runx==0.0.6 tensorboardX coolname

/content
fatal: destination path 'city-surfaces' already exists and is not an empty directory.
/content/city-surfaces


In [12]:
!mkdir -p /content/assets/seg_weights
!cp /content/drive/MyDrive/citysurfaces/*.pth /content/assets/seg_weights/
!ls -lh /content/assets/seg_weights

total 848M
-rw------- 1 root root 551M Oct 29 18:59 block_c_10classes.pth
-rw------- 1 root root 297M Oct 29 18:59 hrnetv2_w48_imagenet_pretrained.pth


In [13]:
!sed -i "s|__C.ASSETS_PATH = ''|__C.ASSETS_PATH = '/content/assets'|" config.py
!sed -i "s|np.int(np.sum|int(np.sum|" network/hrnetv2.py
!sed -i "s|astype(np.bool)|astype(bool)|" utils/f_boundary.py
!sed -i "0,/^import torch$/s//import torch, functools\ntorch.load = functools.partial(torch.load, weights_only=False)/" val.py

In [15]:
import os
from PIL import Image, ImageOps

SRC = '/content/drive/MyDrive/citysurfaces/photos'
DST = '/content/photos_small'
MAX_SIDE = 2048   # longest side in pixels; try 1024 if you still run out of memory

os.makedirs(DST, exist_ok=True)
for name in sorted(os.listdir(SRC)):
    img = Image.open(os.path.join(SRC, name))
    print(name, img.size, end=' -> ')
    img = ImageOps.exif_transpose(img).convert('RGB')
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    img.save(os.path.join(DST, os.path.splitext(name)[0] + '.png'))
    print(img.size)

IMG_2896.JPG (5712, 4284) -> (2048, 1536)
IMG_2897.JPG (5712, 4284) -> (2048, 1536)
IMG_2898.JPG (5712, 4284) -> (2048, 1536)
IMG_2899.JPG (5712, 4284) -> (2048, 1536)
IMG_2900.JPG (5712, 4284) -> (2048, 1536)
IMG_2901.JPG (5712, 4284) -> (2048, 1536)
IMG_2902.JPG (5712, 4284) -> (2048, 1536)
IMG_2903.JPG (5712, 4284) -> (2048, 1536)
IMG_2904.JPG (5712, 4284) -> (2048, 1536)
IMG_2905.JPG (5712, 4284) -> (2048, 1536)
IMG_2906.JPG (5712, 4284) -> (2048, 1536)
IMG_2907.JPG (5712, 4284) -> (2048, 1536)
IMG_2908.JPG (5712, 4284) -> (2048, 1536)
IMG_2909.JPG (5712, 4284) -> (2048, 1536)
IMG_2910.JPG (5712, 4284) -> (2048, 1536)
IMG_2911.JPG (5712, 4284) -> (2048, 1536)
IMG_2912.JPG (5712, 4284) -> (2048, 1536)
IMG_2913.JPG (5712, 4284) -> (2048, 1536)
IMG_2914.JPG (5712, 4284) -> (2048, 1536)
IMG_2915.JPG (5712, 4284) -> (2048, 1536)
IMG_2916.JPG (5712, 4284) -> (2048, 1536)
IMG_2917.JPG (5712, 4284) -> (2048, 1536)
IMG_2918.JPG (5712, 4284) -> (2048, 1536)
IMG_2920.JPG (5712, 4284) -> (2048

In [16]:
!python -m val --dataset citysurfaces --bs_val 1 --eval test \
    --eval_folder /content/photos_small \
    --snapshot /content/assets/seg_weights/block_c_10classes.pth \
    --arch ocrnet.HRNet_Mscale --trunk hrnetv2 \
    --result_dir /content/results

I0000 00:00:1793301082.043032    8602 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793301084.507433    8602 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
None
Using a single GPU.
Torch version: 2.11, 2.11.0+cu130
dataset = citysurfaces
ignore_label = -1
num_classes = 10
Found 43 folder imgs
all imgs 43
Using Per Image based weighted loss
Using Cross Entropy Loss
Loading weights from: checkpoint=/content/assets/seg_weights/block_c_10classes.pth
=> init weights from normal distribution
=> loading pretrained model /content/assets/seg_weights/hrnetv2_w48_imagenet_pretrained.pth
Trunk: hrnetv2
Model params = 72.1M
validating[Iter: 1 / 43]
validating[Iter: 21 / 43]
validating[Iter: 41 / 43]


In [17]:
# Option A: copy to your Drive
!cp -r /content/results /content/drive/MyDrive/citysurfaces/
